# Strategy 3: PPO/DDQN training on Colab CPU

Runs `Strategy 3`'s pipeline: PPO/DDQN training (ASU as a low-probability
training opponent only) -> seat-balanced evaluation against Fixed-A/B/C.

Read `Strategy 3/PLAN.md` first, especially §2 and §9's rule-compliance note.
The MonopolyZero/`monopoly_bench` self-play track was dropped: its `Trainer`
bootstraps the policy head by cross-entropy against ASU's chosen actions
unconditionally on every fresh run, with no way to disable it through the
public API. That is training on ASU's output, which the competition rules
(2026-08-10/11, "ASU'yu birebir output klonlamak yasak") and this repo's own
`CLAUDE.md` both forbid outside the SLM/Gemma track. Two settled points this
notebook assumes:

- **Metric (PLAN.md §1)**: baseline-relative. The number that matters is this
  checkpoint's seat-balanced win rate against Fixed-A/B/C, compared against
  the measured `asu_value_v1` baseline of **72/100** (`Strategy 1/REPO_STUDY_NOTES.md`).
  ASU is never seated as an opponent in the final evaluation cell.
- **ASU's role (PLAN.md §2, corrected)**: opponent seat only, at low
  probability (`--asu-opponent-probability`). Never a training target.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this per parallel Colab Pro tab (a, b, c, d, e, f) so 6 simultaneous
# runs write to different Drive paths instead of clobbering each other.
RUN_TAG = 'a'

DRIVE_ROOT = '/content/drive/MyDrive/DeepRL_Monopoly'
!mkdir -p "$DRIVE_ROOT/artifacts"

## Get the code

Clones from the `Gokturkakman/DeepRL_Monopoly` fork (no write access to
`Darkosxl/DeepRL_Monopoly` upstream), branch `feature/strategy-3-hybrid` --
`Strategy 3` hasn't been merged to `main` yet. If you get a 404 or a tree
without `Strategy 3/`, either the branch was renamed/merged since this was
written (update `BRANCH` below) or it hasn't been pushed yet.

In [ ]:
REPO_URL = 'https://github.com/Gokturkakman/DeepRL_Monopoly.git'  # fork -- no write access to Darkosxl/DeepRL_Monopoly
BRANCH = 'feature/strategy-3-hybrid'  # not merged to main yet
REPO_DIR = '/content/DeepRL_Monopoly/Strategy 3'

!rm -rf /content/DeepRL_Monopoly
!git clone --branch $BRANCH --depth 1 $REPO_URL /content/DeepRL_Monopoly
%cd "$REPO_DIR"
!ls

In [ ]:
# Colab ships torch + numpy preinstalled and both already satisfy this
# repo's requirements.txt (numpy>=1.26) -- no upgrade needed. (An earlier
# version of this cell ran `pip install --upgrade numpy`, which only
# confused the version print below -- the upgraded wheel lands on disk but
# this kernel already had numpy cached from Colab's own startup, so the
# print kept showing the old version until a runtime restart. Training
# cells below spawn fresh `!python` subprocesses regardless, which would
# have picked up the upgrade either way -- but skipping it avoids the
# numba<2.5 dependency-conflict warning Colab's preinstalled numba throws.)
import torch, numpy
print('torch', torch.__version__, 'cuda available (unused here):', torch.cuda.is_available())
print('numpy', numpy.__version__)

## Step 1 -- PPO + DDQN in parallel

The TPU v5e-1 runtime type gives a much bigger vCPU allocation than the plain
CPU runtime -- the game-simulation loop itself won't use the TPU chip
(nothing in this repo does; see PLAN.md's compute notes), but with more
cores available it's worth running the two independent training runs as
separate background processes instead of one after another.

**2026-08-12 session, `--asu-opponent-probability 0.15` history**: raising it
from 0.02 (to make the model actually train against ASU often enough to
matter) repeatedly stalled both PPO and DDQN for 25-30+ minutes with zero
game-window progress, confirmed via `py-spy dump` to be genuinely stuck deep
in `ASU_FROZEN_TEACHER`'s trade-candidate evaluation (`rent_projection` /
`_trade_candidate`), not hung. The `--asu-decision-timeout` guard didn't
save it -- with the same `--seed` across restarts the run kept landing on
the same pathological early trade-negotiation state. Fixed by changing the
seed (escapes that specific trajectory) and lowering the timeout to `0.5`
(tighter cap on worst-case cost per pathological decision). Also found and
fixed a real bug this session: `ASUOpponent.choose_action`'s
`except _DecisionTimeout` sat beside the `finally` that disarms the SIGALRM
timer, not around it -- if the alarm fired in the narrow window between the
try body returning and the finally's own disarm call, the exception came
from inside the finally and the sibling except couldn't catch it, crashing
the process (see `ASU_FROZEN_TEACHER/opponent.py`). Confirmed via a live
crash mid-training (PPO, 400 games in, at the old 0.15/2.0 settings) --
lowering the timeout to 0.5 narrows the margin between ordinary-decision
latency and the timeout, making the race easier to hit, so the fix mattered
more once the timeout was tightened. `--asu-opponent-probability` is left at
**0.02** here (not restored to 0.15) since even the fixed timeout doesn't
bound the *count* of pathological decisions in one game, only the cost of
each -- 0.15 exposure is still a real risk of long stalls.

`--self-play-probability` (DDQN only) is a *separate* mechanism layered on
top of the ASU opponent seat, not instead of it -- past DDQN snapshots as
additional opponents, pure self-play against its own history, no ASU
involvement in that specific piece. DDQN also gets the diagnosed short-run
fix -- default `lr=1e-5` / `target_update_freq=500 games` are tuned for a
10,000-game paper run; a 1000-game run at those defaults stayed at 0% win
rate with correctly-signed rewards throughout (`CLAUDE.md`, "Known-hard
problem").

**DDQN is much slower than PPO per game on this CPU tier** -- not a bug,
confirmed via repeated `py-spy dump` showing genuine `torch.autograd`
backward passes (`agent_ddqn.py:update`), not a stall. DDQN's off-policy
design runs a full gradient step after *every* stored transition once the
replay buffer passes `batch_size` (128), while PPO only updates in batches
every `n_steps=1024`; with hidden_dim=1024 + the Dueling+PER upgrade, DDQN's
per-decision cost balloons once the buffer fills. In one 6-hour Colab CPU
session, PPO reached game ~240 while DDQN never reached its first
`--checkpoint-every 100` threshold. Budget accordingly, or expect DDQN to
need a GPU/paid-tier runtime for real progress.

Python buffers stdout when it isn't a terminal (true for both `!python` and
a `subprocess.Popen` writing to a file), so without `-u` the per-window
progress prints sit in a buffer and don't show up for a long time even
though training is actually progressing -- `python -u` below forces
unbuffered output so the log file updates as it happens, and the wait cell
now tails both logs every minute so you can watch it move instead of
guessing whether it's stuck.


### Colab Pro: 6 parallel tabs, GPU

With Colab Pro you can open this notebook in up to ~6 browser tabs at once,
each connecting to its own runtime -- pick **GPU: T4** in each (Runtime ->
Change runtime type). T4 is enough for this network size (hidden_dim=1024,
small dueling MLP) and burns far fewer compute units than A100/L4, which
matters when running 6 at once. `--device cuda` below actually matters for
DDQN specifically -- its bottleneck (confirmed via `py-spy`) is real
`torch.autograd` backward passes on every stored transition once the replay
buffer fills, which a GPU handles far faster than this CPU tier. ASU's own
decision search is pure Python and unaffected by GPU -- the seed/timeout
mitigation above still applies regardless of runtime.

Suggested 6-tab grid (set `RUN_TAG` in the drive-mount cell above, and the
`GAMES`/`SEED`/`SELF_PLAY_PROBABILITY`/`ASU_OPPONENT_PROBABILITY`/`PPO_LR`/
`DDQN_LR` variables in the cell below, per tab):

| Tab | RUN_TAG | Focus | Change from baseline |
|---|---|---|---|
| 1 | `a` | PPO baseline | none -- control, same settings that got PPO to game ~240 today |
| 2 | `b` | PPO, more self-play | `SELF_PLAY_PROBABILITY='0.30'` |
| 3 | `c` | PPO, more ASU exposure | `ASU_OPPONENT_PROBABILITY='0.06'` (moderate increase now that the stall is seed/timeout-mitigated, not back to 0.15) |
| 4 | `d` | DDQN baseline (GPU) | none -- see if GPU alone fixes the speed problem |
| 5 | `e` | DDQN, more self-play | `SELF_PLAY_PROBABILITY='0.30'` |
| 6 | `f` | PPO baseline, different seed | `SEED='23'` -- variance/robustness replicate of tab 1 |

Watch **`FixedABC%`** in the log, not `Win%` -- that's the deterministic
Fixed-A/B/C number (the scored metric, PLAN.md §1), logged every
`log_every` games now. `Win%` is the noisy in-pool number and can read much
higher without meaning the policy actually plays well.

Note: the scored competition metric is win rate vs. Fixed-A/B/C, not a
head-to-head vs. ASU (PLAN.md §1) -- ASU only occupies an opponent *seat*
during training, at low probability. `asu_value_v1`'s 72/100 vs Fixed-A/B/C
is a reference point, not the thing being optimized against directly.

In [ ]:
import subprocess

PPO_OUT = f'{DRIVE_ROOT}/artifacts/ppo_plus/ppo_hybrid_{RUN_TAG}.pt'
DDQN_OUT = f'{DRIVE_ROOT}/artifacts/ddqn_plus/ddqn_hybrid_{RUN_TAG}.pt'
ppo_log = '/content/ppo_train.log'
ddqn_log = '/content/ddqn_train.log'

# --- Edit these per parallel tab per the 6-variant grid in the markdown above ---
GAMES = '2000'
SEED = '7'
SELF_PLAY_PROBABILITY = '0.15'
ASU_OPPONENT_PROBABILITY = '0.02'
PPO_LR = None   # e.g. '3e-4' to override; None leaves PPOAgent's default
DDQN_LR = '1e-4'

ppo_cmd = [
    'python', '-u', 'tools/train_and_save.py',
    '--algo', 'ppo', '--hybrid',
    '--games', GAMES,
    '--device', 'cuda',
    '--seed', SEED,
    '--checkpoint-every', '100',
    '--opponent-epsilon', '0.1',
    '--opponent-threshold-jitter', '0.2',
    '--held-out-eval-games', '20',
    '--real-eval-games', '20',
    '--asu-opponent-probability', ASU_OPPONENT_PROBABILITY,
    '--asu-decision-timeout', '0.5',
    '--self-play-probability', SELF_PLAY_PROBABILITY,
    '--self-play-pool-size', '8',
    '--self-play-register-every', '200',
    '--out', PPO_OUT,
] + (['--lr', PPO_LR] if PPO_LR else [])
ddqn_cmd = [
    'python', '-u', 'tools/train_and_save.py',
    '--algo', 'ddqn', '--hybrid',
    '--games', GAMES,
    '--device', 'cuda',
    '--seed', SEED,
    '--checkpoint-every', '100',
    '--lr', DDQN_LR,
    '--target-update-freq-steps', '2000',
    '--epsilon-decay', '0.9985',
    '--opponent-epsilon', '0.1',
    '--opponent-threshold-jitter', '0.2',
    '--held-out-eval-games', '20',
    '--real-eval-games', '20',
    '--asu-opponent-probability', ASU_OPPONENT_PROBABILITY,
    '--asu-decision-timeout', '0.5',
    '--self-play-probability', SELF_PLAY_PROBABILITY,
    '--self-play-pool-size', '8',
    '--self-play-register-every', '200',
    '--out', DDQN_OUT,
]

# cwd is explicit rather than inherited from the kernel's current directory --
# if you re-ran the "Get the code" cell after an earlier rm -rf/re-clone, the
# kernel can be left holding a stale cwd handle to a now-deleted directory,
# which makes any inherited-cwd subprocess fail with a getcwd() error before
# it even starts. REPO_DIR (set in that cell) is always the fresh path.
ppo_proc = subprocess.Popen(ppo_cmd, cwd=REPO_DIR, stdout=open(ppo_log, 'w'), stderr=subprocess.STDOUT)
ddqn_proc = subprocess.Popen(ddqn_cmd, cwd=REPO_DIR, stdout=open(ddqn_log, 'w'), stderr=subprocess.STDOUT)
print('RUN_TAG:', RUN_TAG)
print('PPO started, pid', ppo_proc.pid, '-> log:', ppo_log)
print('DDQN started, pid', ddqn_proc.pid, '-> log:', ddqn_log)
print('Run the next cell to wait for both and tail their logs.')

Wait for both background runs, printing a status line plus the last few log
lines from each every minute -- this is where you watch progress game by
game (each `log_every`-games window prints a win-rate line). Safe to
interrupt and re-run -- it only polls the already-running processes, it
doesn't restart them.

In [ ]:
import time

while ppo_proc.poll() is None or ddqn_proc.poll() is None:
    print(f"\n[{time.strftime('%H:%M:%S')}] PPO running={ppo_proc.poll() is None}  DDQN running={ddqn_proc.poll() is None}")
    print('--- PPO log (last 5 lines) ---')
    !tail -n 5 {ppo_log}
    print('--- DDQN log (last 5 lines) ---')
    !tail -n 5 {ddqn_log}
    time.sleep(60)

print('\nPPO exit code:', ppo_proc.returncode)
print('DDQN exit code:', ddqn_proc.returncode)
print('\n--- PPO log (last 30 lines) ---')
!tail -n 30 {ppo_log}
print('\n--- DDQN log (last 30 lines) ---')
!tail -n 30 {ddqn_log}

## Step 2 -- the metric that matters (PLAN.md §1)

Seat-balanced win rate against Fixed-A/B/C, with a Wilson interval
(`tools/evaluate_vs_fixed.py`, built on `ASU_FROZEN_TEACHER.evaluate.evaluate_lineup`
-- the exact function that produced the measured `asu_value_v1` baseline of
**72/100**, so this is the same code path, not just the same statistic).
ASU is not an opponent in this cell -- per PLAN.md §1 this is a
baseline-relative comparison, not a head-to-head against ASU.

In [ ]:
import os

# Self-contained: redefines the same constants as the earlier cells instead
# of trusting kernel state. If you reconnected the runtime and jumped
# straight to Step 2 (drive/repo not remounted this session), the old
# "%cd $REPO_DIR" silently no-op'd on an undefined/stale var and left cwd at
# /content, so tools/evaluate_vs_fixed.py resolved to the wrong path with no
# error until python's own "No such file" -- os.chdir() below fails loudly
# instead if the repo really isn't there. RUN_TAG must match what this tab
# actually trained with (see the drive-mount cell) or you'll evaluate the
# wrong checkpoint.
RUN_TAG = 'a'
DRIVE_ROOT = '/content/drive/MyDrive/DeepRL_Monopoly'
REPO_DIR = '/content/DeepRL_Monopoly/Strategy 3'
PPO_OUT = f'{DRIVE_ROOT}/artifacts/ppo_plus/ppo_hybrid_{RUN_TAG}.pt'
DDQN_OUT = f'{DRIVE_ROOT}/artifacts/ddqn_plus/ddqn_hybrid_{RUN_TAG}.pt'

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

!python -u tools/evaluate_vs_fixed.py \\
  --checkpoint "ppo:$PPO_OUT" \\
  --seeds 25 \\
  --out "$DRIVE_ROOT/artifacts/ppo_plus/eval_vs_fixed_abc_{RUN_TAG}.json"

!python -u tools/evaluate_vs_fixed.py \\
  --checkpoint "ddqn:$DDQN_OUT" \\
  --seeds 25 \\
  --out "$DRIVE_ROOT/artifacts/ddqn_plus/eval_vs_fixed_abc_{RUN_TAG}.json"

Checkpoints and eval JSON all live under `$DRIVE_ROOT` on Drive, so they
survive the Colab VM being recycled. Pull anything back to your laptop by
downloading it from Drive directly -- don't route it through git;
`artifacts/` is gitignored on purpose.